In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
sample=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
sample.to_csv('submission.csv',index=False)


# Milestones


## Milestone 1


In [2]:
# Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv.
# Based on your counts, what is the sum of the occurrences of the most frequent option and 
# the least frequent option?  

import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
 
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
 
freq = df["answer"].value_counts()
print("\n── Q1: Answer frequency ──")
print(freq)
 
most_frequent   = freq.max()
least_frequent  = freq.min()
q1_answer       = most_frequent + least_frequent
print(f"Most frequent count : {most_frequent}")
print(f"Least frequent count: {least_frequent}")
print(f"Sum (Q1 answer)     : {q1_answer}")



Shape: (2000, 8)
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

── Q1: Answer frequency ──
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Most frequent count : 490
Least frequent count: 324
Sum (Q1 answer)     : 814


In [3]:
#After converting the prompt column to lowercase and removing all standard punctuation characters 
#(using Python's string.punctuation), split the text by whitespace. What is the total number of unique 
#words (vocabulary size) across the entire cleaned prompt column of train.csv?  


def clean_text(text):
    """Lowercase + remove standard punctuation."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text
 
df["prompt_clean"] = df["prompt"].apply(clean_text)
 
all_words = []
for text in df["prompt_clean"]:
    all_words.extend(text.split())
 
vocab = set(all_words)
print(f"\n── Q2: Vocabulary size ──")
print(f"Total unique words: {len(vocab)}")


── Q2: Vocabulary size ──
Total unique words: 859


In [5]:
#Using the cleaned prompt from Row ID 1, filter out the standard English stop words 
#using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left 
#in the prompt for Row ID 1 after filtering?  


row1 = df[df["id"] == 1].iloc[0]["prompt_clean"]
tokens_row1 = row1.split()
filtered_row1 = [w for w in tokens_row1 if w not in ENGLISH_STOP_WORDS]
 
print(f"\n── Q3: Row ID 1 after stop-word removal ──")
print(f"Original tokens : {len(tokens_row1)}")
print(f"After filtering : {len(filtered_row1)}")
print(f"Remaining words : {filtered_row1}")


── Q3: Row ID 1 after stop-word removal ──
Original tokens : 22
After filtering : 13
Remaining words : ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


In [6]:
#Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined 
#text of the prompts and options in train.csv. What is the exact total number of feature 
#columns (vocabulary size) generated by the vectorizer?  

option_cols = ["A", "B", "C", "D", "E"]
 
corpus = []
for _, row in df.iterrows():
    for col in ["prompt"] + option_cols:
        if not pd.isna(row[col]):
            corpus.append(str(row[col]))
 
vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(corpus)
 
vocab_size = len(vectorizer.vocabulary_)
print(f"\n── Q4: TF-IDF vocabulary size ──")
print(f"Total feature columns: {vocab_size}")


── Q4: TF-IDF vocabulary size ──
Total feature columns: 2762


In [8]:
#Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the 
#prompt and option A strictly for Row ID 1. What is the resulting similarity score? 
#(Round to 4 decimal places).  


row1_data = df[df["id"] == 1].iloc[0]

prompt_vec  = vectorizer.transform([str(row1_data["prompt"])])
optionA_vec = vectorizer.transform([str(row1_data["A"])])
 
sim_q5 = cosine_similarity(prompt_vec, optionA_vec)[0][0]
print(f"\n── Q5: Cosine similarity (Row 1, prompt vs A) ──")
print(f"Similarity: {round(sim_q5, 4)}")


── Q5: Cosine similarity (Row 1, prompt vs A) ──
Similarity: 0.2328


In [9]:
#Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity 
#between the prompt and each of its 5 options .  Then calculate the percentage of instances 
#where the option with the highest cosine similarity matches the correct answer.   



def best_option_by_tfidf(row):
    p_vec = vectorizer.transform([str(row["prompt"])])
    sims  = {}
    for opt in option_cols:
        if not pd.isna(row[opt]):
            o_vec    = vectorizer.transform([str(row[opt])])
            sims[opt] = cosine_similarity(p_vec, o_vec)[0][0]
    return max(sims, key=sims.get)
 
df["predicted_best"] = df.apply(best_option_by_tfidf, axis=1)
accuracy = (df["predicted_best"] == df["answer"]).mean() * 100
 
print(f"\n── Q6: TF-IDF top-1 accuracy ──")
print(f"Accuracy: {round(accuracy, 2)}%")


── Q6: TF-IDF top-1 accuracy ──
Accuracy: 13.7%


In [11]:
#If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  


def average_precision_at_k(predicted, ground_truth, k=3):
    """
    predicted    : list of predicted labels (up to k)
    ground_truth : single correct label (string)
    """
    score = 0.0
    hits  = 0
    for i, pred in enumerate(predicted[:k]):
        if pred == ground_truth:
            hits  += 1
            score += hits / (i + 1)
    return score


ap_q7 = average_precision_at_k(["C", "A", "B"], "C")
print(f"\n── Q7: MAP@3 for [C,A,B] with GT=C ──")
print(f"AP@3: {ap_q7}")


── Q7: MAP@3 for [C,A,B] with GT=C ──
AP@3: 1.0


In [12]:
#If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

ap_q8 = average_precision_at_k(["D", "B", "E"], "B")
print(f"\n── Q8: MAP@3 for [D,B,E] with GT=B ──")
print(f"AP@3: {ap_q8}")
 


── Q8: MAP@3 for [D,B,E] with GT=B ──
AP@3: 0.5


In [13]:
#The Majority Class Baseline: Find the most frequent correct answer in the training set 
#(using your data from Q1). Make a static prediction for every single row where that 
#most frequent answer is your 1st guess, followed by the second most frequent, 
#and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline 
#on train.csv?


top3_answers = freq.index[:3].tolist()  # e.g. ['B', 'A', 'C']
print(f"\n── Q9: Majority class baseline ──")
print(f"Top-3 most frequent answers used as predictions: {top3_answers}")
 
map3_majority = np.mean([
    average_precision_at_k(top3_answers, gt)
    for gt in df["answer"]
])
print(f"Majority baseline MAP@3: {round(map3_majority, 4)}")


── Q9: Majority class baseline ──
Top-3 most frequent answers used as predictions: ['B', 'C', 'A']
Majority baseline MAP@3: 0.4212


In [14]:
#The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. 
#For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. 
#Sort these options from highest similarity to lowest to form your top 3 predictions. 
#What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

def tfidf_top3(row):
    p_vec = vectorizer.transform([str(row["prompt"])])
    sims  = {}
    for opt in option_cols:
        if not pd.isna(row[opt]):
            o_vec     = vectorizer.transform([str(row[opt])])
            sims[opt] = cosine_similarity(p_vec, o_vec)[0][0]
    # Sort options by similarity desc, take top 3
    ranked = sorted(sims, key=sims.get, reverse=True)[:3]
    return ranked
 
print(f"\n── Q10: TF-IDF Pipeline MAP@3 ──")
ap_scores = []
for _, row in df.iterrows():
    preds = tfidf_top3(row)
    ap_scores.append(average_precision_at_k(preds, row["answer"]))
 
map3_tfidf = np.mean(ap_scores)
print(f"TF-IDF pipeline MAP@3: {round(map3_tfidf, 4)}")


── Q10: TF-IDF Pipeline MAP@3 ──
TF-IDF pipeline MAP@3: 0.3119


## Milestone 2

In [2]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

torch.manual_seed(0)


In [3]:
ds = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')['train']

def add_combined(example):
    example['combined_text'] = example['prompt'] + " " + example['A']
    return example

ds = ds.map(add_combined)

combined_text_51 = ds[51]['combined_text']
length_51 = len(combined_text_51)
print("combined_text[51]:", combined_text_51)
print("Character length:", length_51)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

combined_text[51]: Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
Character length: 614


In [4]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
vocab_size = tokenizer.vocab_size
print("Vocab size:", vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size: 30522


In [6]:
sep_token_id = tokenizer.sep_token_id
print("[SEP] token id:", sep_token_id)

[SEP] token id: 102


In [8]:
prompt_texts = [str(p) if p is not None else "" for p in ds['prompt']]

tokenized_prompts = tokenizer(
    prompt_texts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

input_ids_shape = tuple(tokenized_prompts['input_ids'].shape)
print("input_ids shape:", input_ids_shape)


input_ids shape: (2000, 128)


In [9]:

hidden_size = 768
num_heads = 12
head_dim = hidden_size // num_heads
print("Each attention head dimensionality:", head_dim)

Each attention head dimensionality: 64


In [11]:
def s(x):
    """Coerce a possibly-None CSV cell to a plain string."""
    return str(x) if x is not None else ""


model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

prompt_0 = s(ds[0]['prompt'])
inputs_0 = tokenizer(prompt_0, return_tensors='pt')

with torch.no_grad():
    outputs_0 = model(**inputs_0)

last_hidden_state = outputs_0.last_hidden_state
print("last_hidden_state shape:", tuple(last_hidden_state.shape))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


last_hidden_state shape: (1, 31, 768)


In [12]:
cls_embedding = last_hidden_state[0, 0, :]
cls_sum_first5 = cls_embedding[:5].sum().item()
print("First 5 CLS values:", cls_embedding[:5].tolist())
print("Sum (rounded 4dp):", round(cls_sum_first5, 4))

First 5 CLS values: [-0.4676641821861267, -0.07544498145580292, -0.20190085470676422, -0.007064265664666891, -0.4480222165584564]
Sum (rounded 4dp): -1.2001


In [13]:
attn_model = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
attn_model.eval()

text = "Light-ion fusion is a technique."
attn_inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    attn_outputs = attn_model(**attn_inputs)

attentions = attn_outputs.attentions  # tuple: (num_layers, batch, heads, seq, seq)
last_layer_attn = attentions[-1]      # (batch, heads, seq, seq)

tokens = tokenizer.convert_ids_to_tokens(attn_inputs['input_ids'][0])
print("Tokens:", tokens)

fusion_idx = tokens.index('fusion')
print("Index of 'fusion' token:", fusion_idx)

# [batch=0, head=0, from_token=0 (CLS), to_token=fusion_idx]
cls_to_fusion_weight = last_layer_attn[0, 0, 0, fusion_idx].item()
print("Attention weight (CLS -> fusion), rounded 4dp:", round(cls_to_fusion_weight, 4))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Index of 'fusion' token: 4
Attention weight (CLS -> fusion), rounded 4dp: 0.1025


In [14]:
sbert_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

prompt_0_text = s(ds[0]['prompt'])
option_b_0 = s(ds[0]['B'])

emb_prompt = sbert_model.encode(prompt_0_text, convert_to_tensor=True)
emb_b = sbert_model.encode(option_b_0, convert_to_tensor=True)

sim_score = util.cos_sim(emb_prompt, emb_b).item()
print("Cosine similarity (prompt vs Option B, row 0), rounded 4dp:", round(sim_score, 4))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cosine similarity (prompt vs Option B, row 0), rounded 4dp: 0.7658


In [15]:
options_cols = ['A', 'B', 'C', 'D', 'E']

def mapk_score(true_answers, top3_predictions):
    """MAP@3 across the dataset."""
    scores = []
    for true_ans, preds in zip(true_answers, top3_predictions):
        if true_ans in preds:
            rank = preds.index(true_ans) + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
    return float(np.mean(scores))

prompts_all = [s(p) for p in ds['prompt']]
true_answers = ds['answer']

# ---- Pipeline 1: TF-IDF cosine similarity ----
tfidf_top3 = []
for i, prompt_text in enumerate(prompts_all):
    option_texts = [s(ds[i][c]) for c in options_cols]
    corpus = [prompt_text] + option_texts
    vec = TfidfVectorizer().fit_transform(corpus)
    sims = cosine_similarity(vec[0:1], vec[1:]).flatten()
    ranked = [options_cols[j] for j in np.argsort(-sims)]
    tfidf_top3.append(ranked[:3])

tfidf_map3 = mapk_score(true_answers, tfidf_top3)
print("TF-IDF pipeline MAP@3:", round(tfidf_map3, 4))

# ---- Pipeline 2: MiniLM sentence-embedding cosine similarity ----
minilm_top3 = []
for i, prompt_text in enumerate(prompts_all):
    option_texts = [s(ds[i][c]) for c in options_cols]
    emb_p = sbert_model.encode(prompt_text, convert_to_tensor=True)
    emb_opts = sbert_model.encode(option_texts, convert_to_tensor=True)
    sims = util.cos_sim(emb_p, emb_opts).flatten().cpu().numpy()
    ranked = [options_cols[j] for j in np.argsort(-sims)]
    minilm_top3.append(ranked[:3])

minilm_map3 = mapk_score(true_answers, minilm_top3)
print("MiniLM pipeline MAP@3 (final answer):", round(minilm_map3, 4))

# ---- Comparison: TF-IDF missed, MiniLM caught ----
count_tfidf_missed_minilm_caught = sum(
    1 for true_ans, tfidf_p, minilm_p in zip(true_answers, tfidf_top3, minilm_top3)
    if true_ans not in tfidf_p and true_ans in minilm_p
)
print("Count where TF-IDF Top-3 missed but MiniLM Top-3 caught:", count_tfidf_missed_minilm_caught)


TF-IDF pipeline MAP@3: 0.3453
MiniLM pipeline MAP@3 (final answer): 0.4231
Count where TF-IDF Top-3 missed but MiniLM Top-3 caught: 522


In [17]:
zs_pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row1 = ds[1]
candidate_labels = [s(row1['A']), s(row1['B']), s(row1['C'])]

zs_result = zs_pipe(s(row1['prompt']), candidate_labels)
top_score = zs_result['scores'][0]
print("Zero-shot result:", zs_result)
print("Top-ranked option probability, rounded 4dp:", round(top_score, 4))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Zero-shot result: {'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient t

In [18]:
zs_result_multi = zs_pipe(row1['prompt'], candidate_labels, multi_label=True)

sum_softmax = sum(zs_result['scores'])
sum_sigmoid = sum(zs_result_multi['scores'])

abs_diff = abs(sum_softmax - sum_sigmoid)
print("Sum (softmax):", round(sum_softmax, 4))
print("Sum (independent sigmoids):", round(sum_sigmoid, 4))
print("Absolute difference:", round(abs_diff, 4))


Sum (softmax): 1.0
Sum (independent sigmoids): 0.0005
Absolute difference: 0.9995


In [22]:
gen_pipe = pipeline("text-generation", model="google/flan-t5-small")

row0 = ds[0]
gen_prompt = (
    f"Question: {str(row0['prompt'])}. "
    f"Is the correct answer A: {str(row0['A'])} or B: {str(row0['B'])}? "
    f"Answer with just the letter A or B."
)

gen_output = gen_pipe(gen_prompt, max_new_tokens=5)
print("Prompt sent:", gen_prompt)
print("Model output:", gen_output[0]['generated_text'])

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

Prompt sent: Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.
Model output: Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Marti